# Reviewing craft-kit requests at a library

A library lends craft supplies to volunteer hosts who run scheduled
activities. Hosts can pick up small kits for drop-in activities or
workshop packs for groups. The library wants to review repeat pickup
tickets, unusually frequent requests, and unusually expensive or large
kits. It also wants to tell when a change belongs to one host and when
it follows a change to the program calendar.

This example generates a roster, a schedule of eligible dates, and
kit requests across four programs and two session times. Midyear,
Gallery afternoon sessions move toward weekends. That change also
occurs in the control run, which has no planted individual cases.

Polars shapes the inputs. Onceover decides which values are isolated,
which rows repeat, and which request mixes change.

Row duplicates and numeric outliers are in the default detector set. This case requests them explicitly to focus the reports. Entity fingerprints runs on request.

In [1]:
from datetime import date, timedelta
from random import Random

import onceover
import polars as pl

START = date(2025, 1, 6)
MIDPOINT = START + timedelta(weeks=26)
PROGRAMS = ("Story Lab", "Paper Arts", "Makerspace", "Gallery")
SESSIONS = ("Morning", "Afternoon")
WEEKDAYS = ("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")
CASES = {
    "H003": "High request rate",
    "H036": "Repeated pickup tickets",
    "H070": "Persistently expensive small kits",
    "H116": "Switch to weekend requests",
    "H172": "Persistently large workshop packs",
    "H214": "Concentration on Fridays",
    "H245": "Single expensive small kit",
}


def make_library(*, planted=True):
    hosts, schedule, requests = [], [], []
    for p, program in enumerate(PROGRAMS):
        for s, session in enumerate(SESSIONS):
            for i in range(64):
                batch, position = divmod(i, 32)
                number = (8 * batch + 2 * p + s) * 32 + position + 1
                host = f"H{number:03}"
                rng = Random(448_000 + number)
                base_rate = 0.36 + 0.025 * p + 0.06 * s + rng.uniform(-0.025, 0.025)
                unit_cost = (14.8, 16.8, 16.3, 15.4)[p] + 1.2 * s + rng.uniform(-0.7, 0.7)
                hosts.append((host, program, session))
                host_requests = []
                high_ticket_added = False

                for offset in range(52 * 7):
                    day = START + timedelta(days=offset)
                    weekend = day.weekday() >= 5
                    period = 1 if day < MIDPOINT else 2
                    weekday = WEEKDAYS[day.weekday()]
                    scheduled_rate = (
                        (0.28, 0.40, 0.60, 0.33)[p] + 0.14 * s
                        if weekend else 0.54 + 0.09 * s
                    )
                    if (program, session) == ("Gallery", "Afternoon"):
                        # Keep the calendar change in the control too.
                        early = (0.75, 0.10)
                        late = (0.28, 0.92)
                        scheduled_rate = (early if period == 1 else late)[weekend]

                    # Keep the draws paired even where a plant changes a decision.
                    schedule_draw, request_draw, pack_draw = [rng.random() for _ in range(3)]
                    size_draw, cost_draw = rng.randint(0, 3), rng.uniform(-2.7, 2.7)
                    if schedule_draw >= scheduled_rate:
                        continue
                    schedule.append((host, day, weekday, period))
                    rate = base_rate + (0.04 if weekday == "Fri" else 0)
                    if planted and host == "H003":
                        rate = 0.96
                    elif planted and host == "H116":
                        preferred = weekend if period == 2 else not weekend
                        rate = (0.94 if period == 2 else 0.88) if preferred else 0.03
                    elif planted and host == "H214":
                        rate = 0.98 if weekday == "Fri" else 0.13
                    if request_draw >= rate:
                        continue

                    workshop = pack_draw < (0.28 if p == 2 else 0.15) + 0.08 * s
                    size = (5 + size_draw if p == 2 else 3 + size_draw) if workshop else 1
                    price = unit_cost + cost_draw - int(workshop)
                    if planted and host == "H070" and not workshop:
                        price += 24
                    if planted and host == "H172" and workshop:
                        size += 12
                    if planted and host == "H245" and not workshop and not high_ticket_added:
                        price, high_ticket_added = 148.0, True

                    request = {
                        "ticket": f"T{len(requests) + 1:06}", "host": host,
                        "program": program, "session": session, "date": day,
                        "weekday": weekday, "day_type": "Weekend" if weekend else "Weekday",
                        "period": period, "kit": "Workshop pack" if workshop else "Small kit",
                        "pieces": size, "cost": round(price * size, 2),
                    }
                    requests.append(request)
                    host_requests.append(request)

                if planted and host == "H036":
                    requests.extend(host_requests[:8])

    return (
        pl.DataFrame(hosts, schema=["host", "program", "session"], orient="row"),
        pl.DataFrame(schedule, schema=["host", "date", "weekday", "period"], orient="row"),
        pl.DataFrame(requests).with_row_index("request_id", offset=1).sort(["host", "date", "request_id"]),
    )

In [2]:
hosts, schedule, requests = make_library()
print(f"{hosts.height:,} hosts on {schedule.height:,} scheduled host-days")
print(f"{requests.height:,} recorded requests from {START} through {START + timedelta(days=363)}")
print(f"Period 2 starts on {MIDPOINT}")

512 hosts on 101,803 scheduled host-days
44,403 recorded requests from 2025-01-06 through 2026-01-04
Period 2 starts on 2025-07-07


## Keep repeat tickets separate from repeat activity

A request can repeat even though the system assigned it a fresh
ID. Compare the ticket and its attributes without that ID. Use
the report's rows method to get the repeated records. Ticket IDs
identify pickups in this fixture, so keep one row per ticket for
the activity and cost comparisons.

In [3]:
def ticket_audit(requests):
    comparable = requests.select("host", "ticket", "date", "kit", "pieces", "cost")
    report = onceover.profile(comparable, detectors=["row_duplicates"])
    duplicate_hosts = set()
    for finding in report.findings:
        repeated = report.rows(finding["id"], comparable)
        duplicate_hosts.update(repeated["host"])
        print(finding["claim"])
        print(f"{repeated['ticket'].n_unique()} affected tickets for {', '.join(sorted(duplicate_hosts))}")
    return duplicate_hosts


duplicate_hosts = ticket_audit(requests)
pickups = requests.unique(subset=["ticket"], maintain_order=True)

8 rows repeat earlier values across the compared columns (8 groups, largest 2)
8 affected tickets for H036


## Compare rates and sizes within peer groups

A scheduled day is an opportunity to request a kit. Divide distinct
request dates by scheduled dates. For each weekday, also compare its
share of request dates with its share of scheduled dates. This makes
a preference for Fridays easier to distinguish from simply being
scheduled on Fridays.

Compare hosts within the same program and session time. Makerspace
packs are generally larger. Keep the median cost of small kits separate
from the median number of pieces in workshop packs. Require at least
15 small kits or 10 workshop packs for the corresponding host median.
For one unusually expensive small kit, scan the pickups directly.

In [4]:
def host_metrics(hosts, schedule, pickups):
    opportunities = schedule.group_by("host").agg(
        pl.len().alias("scheduled"),
        *[(pl.col("weekday") == d).sum().alias(f"scheduled_{d}") for d in WEEKDAYS],
    )
    activity = pickups.group_by("host").agg(
        pl.col("date").n_unique().alias("request_days"),
        *[
            pl.col("date").filter(pl.col("weekday") == d).n_unique().alias(f"requested_{d}")
            for d in WEEKDAYS
        ],
        (pl.col("kit") == "Small kit").sum().alias("small_kits"),
        (pl.col("kit") == "Workshop pack").sum().alias("workshop_packs"),
        pl.col("cost").filter(pl.col("kit") == "Small kit").median().alias("median_small_kit_cost"),
        pl.col("pieces").filter(pl.col("kit") == "Workshop pack").median().alias("median_workshop_pieces"),
    )
    counts = ["request_days", "small_kits", "workshop_packs", *[f"requested_{d}" for d in WEEKDAYS]]
    return (
        hosts.join(opportunities, on="host").join(activity, on="host", how="left")
        .with_columns(pl.col(counts).fill_null(0))
        .with_columns(
            (100 * pl.col("request_days") / pl.col("scheduled")).round(2).alias("request_days_pct"),
            *[
                pl.when(pl.col("request_days") > 0).then(100 * (
                    pl.col(f"requested_{d}") / pl.col("request_days")
                    - pl.col(f"scheduled_{d}") / pl.col("scheduled")
                )).round(2).alias(f"{d}_excess_pp")
                for d in WEEKDAYS
            ],
        )
        .sort("host")
    )


def extremes(frame, metric):
    source = frame.filter(pl.col(metric).is_not_null())
    frame = source.select("host", metric)
    report = onceover.profile(
        frame, detectors=["outliers"],
        hints={"roles": {"host": "identifier", metric: "measure"}},
    )
    for finding in report.select(type="extreme_value", column=metric):
        evidence = finding["evidence"][0]["metrics"]
        for row in report.rows(finding["id"], frame).iter_rows(named=True):
            ticket = source["ticket"][row["__row"]] if "ticket" in source.columns else None
            yield row["host"], row[metric], evidence["side"], evidence["bulk_edge"], ticket


def numeric_audit(hosts, schedule, pickups):
    metrics = host_metrics(hosts, schedule, pickups)
    findings = []
    for (program, session), cohort in sorted(
        metrics.partition_by(["program", "session"], as_dict=True).items()
    ):
        small_kits = pickups.filter(
            (pl.col("program") == program)
            & (pl.col("session") == session)
            & (pl.col("kit") == "Small kit")
        )
        checks = [
            (cohort, "request_days_pct"),
            (cohort.filter(pl.col("small_kits") >= 15), "median_small_kit_cost"),
            (cohort.filter(pl.col("workshop_packs") >= 10), "median_workshop_pieces"),
            (small_kits, "cost"),
            *[(cohort, f"{d}_excess_pp") for d in WEEKDAYS],
        ]
        for frame, metric in checks:
            for host, value, side, edge, ticket in extremes(frame, metric):
                findings.append((host, f"{program} / {session}", metric, value, side, edge, ticket))
    return sorted(findings)


numeric = numeric_audit(hosts, schedule, pickups)
for host, cohort, metric, value, side, edge, ticket in numeric:
    bound = "at most" if side == "high" else "at least"
    ticket_label = f" (ticket {ticket})" if ticket else ""
    print(f"{host} · {cohort} · {metric}: {value:g}; peers {bound} {edge:.4g}{ticket_label}")

H003 · Story Lab / Morning · request_days_pct: 96.36; peers at most 46.58
H070 · Paper Arts / Morning · median_small_kit_cost: 41.12; peers at most 18.27
H172 · Makerspace / Afternoon · median_workshop_pieces: 19; peers at most 7
H214 · Gallery / Morning · Fri_excess_pp: 41.56; peers at most 9.91
H214 · Gallery / Morning · request_days_pct: 24.87; peers at least 35.26
H245 · Gallery / Afternoon · cost: 148; peers at most 19.91 (ticket T040315)


The peer bound is the edge of the comparison values Onceover used.
The finding also records the gap to that edge relative to the
comparison group's central spread.

An empty result from a small group needs a coverage check. Try the
same numerical scan on 12 hosts from a single peer group.

In [5]:
pilot = host_metrics(hosts, schedule, pickups).filter(
    (pl.col("program") == "Story Lab") & (pl.col("session") == "Morning")
).head(12)
report = onceover.profile(
    pilot.select("host", "request_days_pct"), detectors=["outliers"],
    hints={"roles": {"host": "identifier", "request_days_pct": "measure"}},
)
print(report.coverage["outliers"]["detail"])

0 numeric columns scanned against 0 conditioning columns; 0 surprise findings; 0 categorical columns scanned in 0 contexts up to 3 conditions; 0 categorical surprise findings; frame-role target exclusions: host (identifier)


## Discover possible peer hierarchies

A host's program and session time form a peer group. A program can
be a fallback when there are too few hosts in one session time.
Create a key for the program and session combination, then ask
Onceover for candidate hierarchy hints.

The mappings can support more than one path. The schedule also
determines a weekday-to-weekend mapping. That is a valid rollup,
but it describes the outcome being compared here. Review the
candidates and use the host path ending in program.

In [6]:
def fingerprint_rows(pickups):
    return pickups.select(
        "host", "program", "session", "period", "weekday", "day_type", "kit", "cost",
        pl.concat_str("program", "session", separator=" / ").alias("program_session"),
    )


mix_hints = {
    "entity": "host", "ordered_axis": "period",
    "roles": {c: "dimension" for c in (
        "weekday", "day_type", "kit", "program_session", "program", "session",
    )} | {"cost": "measure"},
}
spec = onceover.infer_spec(fingerprint_rows(pickups), hints=mix_hints)
suggestions = [entry["levels"] for entry in spec["hierarchy_candidates"]]
for path in suggestions:
    print(" → ".join(path))
peer_paths = [path for path in suggestions if path[0] == "host" and path[-1] == "program"]

host → program_session → program
host → program_session → session
weekday → day_type


## Separate individual changes from calendar changes

Pass the selected hierarchy as a hint. Onceover assigns the peer
groups and pools at a supported parent if the finer group has too
little data. Compare pickup counts and supply spend over the two
periods. The evidence identifies the selected group and the share
before and after the boundary.

In [7]:
def mix_audit(pickups):
    individual, program_wide = [], []
    rows = fingerprint_rows(pickups)
    for basis, weights in (("pickups", []), ("spend", ["cost"])):
        report = onceover.profile(
            rows, detectors=["entity_fingerprints"],
            hints={**mix_hints, "weights": weights, "hierarchies": peer_paths},
        )
        for finding in report.select(type="joint_distribution_change"):
            metrics = finding["evidence"][0]["metrics"]
            for change in metrics["entity_shifts"]:
                individual.append((f"{change['peer_group']['value']} · {basis}", change))
            for change in metrics["population_shifts"]:
                program_wide.append((f"{change['peer_group']['value']} · {basis}", change))
    return individual, program_wide


changes, shared = mix_audit(pickups)
for cohort, change in changes:
    print(
        f"{change['entity']} · {cohort} · {change['level']} · period {change['period']}: "
        f"{change['before_share']:.0%} → {change['after_share']:.0%} "
        f"(peers {change['peer_before_share']:.0%} → {change['peer_after_share']:.0%})"
    )
for cohort, change in shared:
    print(
        f"Shared · {cohort} · {change['level']} · period {change['period']}: "
        f"{change['before_share']:.0%} → {change['after_share']:.0%} "
        f"across {change['matched_entities']} matched hosts"
    )

H116 · Paper Arts / Afternoon · pickups · Weekday · period 2: 100% → 11% (peers 75% → 76%)
H116 · Paper Arts / Afternoon · pickups · Sat · period 2: 0% → 50% (peers 13% → 12%)
H116 · Paper Arts / Afternoon · spend · Weekend · period 2: 0% → 92% (peers 26% → 23%)
Shared · Gallery / Afternoon · pickups · Weekday · period 2: 95% → 44% across 64 matched hosts
Shared · Gallery / Afternoon · pickups · Sun · period 2: 3% → 28% across 64 matched hosts
Shared · Gallery / Afternoon · spend · Weekend · period 2: 5% → 58% across 64 matched hosts
Shared · Gallery / Afternoon · spend · Sun · period 2: 3% → 30% across 64 matched hosts


Compare the observed shifts with the schedule before interpreting
them. Count scheduled weekend dates for H116 and for the Gallery
afternoon cohort, using the same period boundary.

In [8]:
gallery_afternoon = hosts.filter(
    (pl.col("program") == "Gallery") & (pl.col("session") == "Afternoon")
)["host"]

for label, subset in (
    ("H116", schedule.filter(pl.col("host") == "H116")),
    ("Gallery / Afternoon", schedule.filter(pl.col("host").is_in(gallery_afternoon.to_list()))),
):
    print(label)
    by_period = subset.group_by("period").agg(
        pl.len().alias("scheduled"),
        pl.col("weekday").is_in(["Sat", "Sun"]).sum().alias("weekends"),
    ).sort("period")
    for period, scheduled, weekends in by_period.iter_rows():
        print(f"  Period {period}: {weekends:,}/{scheduled:,} scheduled dates were weekends ({weekends / scheduled:.0%})")

H116
  Period 1: 27/108 scheduled dates were weekends (25%)
  Period 2: 34/121 scheduled dates were weekends (28%)
Gallery / Afternoon
  Period 1: 326/6,633 scheduled dates were weekends (5%)
  Period 2: 3,056/5,404 scheduled dates were weekends (57%)


H116's weekend request share rose sharply even though the host's schedule changed little. For Gallery afternoons, the schedule and the pickup mix moved toward weekends together. Onceover found a shared shift there and did not flag individual hosts for it.

## Run the same audit on a control

Turn off the individual plants while keeping the calendar change.
Count unique hosts flagged across the three scans. A single shift can
produce several findings, so an alert count can overstate the number
of people involved.

In [9]:
control_hosts, control_schedule, control_requests = make_library(planted=False)
control_pickups = control_requests.unique(subset=["ticket"], maintain_order=True)
control_numeric = numeric_audit(control_hosts, control_schedule, control_pickups)
control_changes, control_shared = mix_audit(control_pickups)

flagged = duplicate_hosts | {row[0] for row in numeric} | {r["entity"] for _, r in changes}
control_flagged = (
    ticket_audit(control_requests)
    | {row[0] for row in control_numeric}
    | {r["entity"] for _, r in control_changes}
)
print(f"Planted hosts recovered: {len(flagged & CASES.keys())} of {len(CASES)}")
print(f"Other hosts flagged: {len(flagged - CASES.keys())}")
print(f"Control run: {len(control_flagged)} hosts flagged")
print("Control shared changes:", ", ".join(sorted({key for key, _ in control_shared})))

Planted hosts recovered: 7 of 7
Other hosts flagged: 0
Control run: 0 hosts flagged
Control shared changes: Gallery / Afternoon · pickups, Gallery / Afternoon · spend


A repeated ticket gives the reviewer a set of records to check.
A high median cost calls for looking at what the kit contained.
A shift in mix calls for checking the calendar first.

One seeded control checks what this example recovers. It cannot
estimate the false-alert rate on a real program. Review coverage
when a cohort is small or sparse. See
[entity fingerprints](../methodology/entity_fingerprints.ipynb)
for the assumptions behind the mix comparisons and
[extreme values](../methodology/extreme_values.ipynb) for the numeric
isolation test.